# Inspect external context for the frozen profiles

**Inherited contract:** the fixed national profiles, their uncertainty signals and the completed robustness envelope.

**Purpose:** inspect selected population, workforce, place and patient-experience authorities without allowing contextual variables to redefine the frozen activity profiles.

This notebook validates and displays included machine-readable evidence. Upstream source acquisition and derivation are documented separately and are not presented as being rebuilt here.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_PATH = ROOT / 'configs' / 'reference_apr2025_mar2026.json'
from gpap2.config import load_config
REFERENCE_CONFIG = load_config(CONFIG_PATH)
AUTHORITY_MANIFEST = REFERENCE_CONFIG.resolve(REFERENCE_CONFIG.authority_checksum_file)
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 6)


In [2]:
import pandas as pd
from gpap2.io import validate_authority_file

paths = {
    'profile_summary': ROOT / 'outputs' / 'tables' / 'numeric_profile_descriptive_summary.csv',
    'gpps': ROOT / 'outputs' / 'tables' / 'gpps_precision_profile_summary.csv',
    'marginal_effects': ROOT / 'outputs' / 'tables' / 'context_multinomial_average_marginal_effects.csv',
    'narratives': ROOT / 'outputs' / 'tables' / 'neutral_profile_narratives.csv',
}
for path in paths.values():
    validate_authority_file(path, AUTHORITY_MANIFEST)
tables = {name: pd.read_csv(path) for name, path in paths.items()}
pd.DataFrame({'table': list(tables), 'rows': [len(frame) for frame in tables.values()]})

,table,rows
0,profile_summary,45
1,gpps,15
2,marginal_effects,15
3,narratives,3


In [3]:
from gpap2.profile_labels import PROFILE_SHORT_LABELS

observed_labels = tables['narratives'].set_index('frozen_profile_number')['approved_descriptive_label']
for profile, label in PROFILE_SHORT_LABELS.items():
    assert observed_labels.loc[profile].casefold() == label.casefold()
tables['narratives'][['frozen_profile_number', 'approved_descriptive_label', 'national_n', 'permitted_interpretation', 'interpretation_boundary']]

,frozen_profile_number,approved_descriptive_label,national_n,permitted_interpretation,interpretation_boundary
0,1,"Profile 1: lower recorded activity, higher DNA...",1753,A recurring multivariable pattern of recorded ...,"Does not indicate total demand, unmet need, wo..."
1,2,"Profile 2: higher face-to-face share, longer d...",2312,A recurring multivariable pattern of recorded ...,"Does not indicate total demand, unmet need, wo..."
2,3,"Profile 3: higher recorded activity, higher sa...",2002,A recurring multivariable pattern of recorded ...,"Does not indicate total demand, unmet need, wo..."


## Interpretation contract

Patient experience, workforce, population, deprivation, rurality and geography are practice-level external evidence. Denominators differ by measure. Associations can describe context and test coherence; they cannot establish patient-level mechanisms, performance rankings or causal effects.

**Handover:** contextual findings pass to the geography stage and then to the final claim-to-evidence synthesis.